# Kolos: dish recognition — CLIP embeddings on Google Colab

Step 3b of `ml/foodphoto` (see `ml/README.md`): runs three CLIP image encoders over the same
photos as the trained CNN and saves the vectors to Drive. The comparison itself runs locally
(`python -m ml.foodphoto.clip_analyze`), without a GPU.

**Before the run:** *Runtime → Change runtime type → T4 GPU.* The data cache made by the training
notebook (`prepared.tar` on Drive) is reused, so step 4 takes a few minutes.

**Every session:** *Runtime → Run all.* A finished part is skipped, so after a disconnect just run
everything again.

**At the end:** download `My Drive/Kolos/foodphoto/runs/clip_b32`, `clip_b16` and `clip_l14` and
put them into `ml/models/foodphoto/` locally (~350 MB together).

In [ ]:
# 1. GPU check: a T4 (or better) should be listed.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (persistent storage) and paths.
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kolos/foodphoto'
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
os.environ['FOODPHOTO_DATA'] = '/content/data'          # images: fast local disk
os.environ['FOODPHOTO_RUNS'] = f'{DRIVE}/runs'          # checkpoints and results: Drive
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Code from GitHub (branch phase-f2).
%cd /content
!test -d kolos && git -C kolos pull -q || git clone -q --depth 1 --branch phase-f2 https://github.com/dkrainii/course_project.git kolos
%cd /content/kolos
!git log --oneline -1

In [ ]:
# 4. Data. First session: download Food-101 (~5 GB), resize everything, cache the result on
#    Drive (~2 GB). Later sessions only unpack the cache (~2 min).
#    The last line must list 'ua': 879; otherwise read the message above it.
#    Runs in the notebook process, so an error stops "Run all" here.
from pathlib import Path
from ml.foodphoto import colab_data
colab_data.main(Path(DRIVE), workers=4)

In [ ]:
# 5. CLIP library (OpenAI weights are downloaded on first use).
!pip install -q open_clip_torch==4.2.2

In [ ]:
# 6. ViT-B/32: the smallest CLIP (~5 min).
!python -m ml.foodphoto.clip_embed --model clip_b32 --workers 2

In [ ]:
# 7. ViT-B/16 (~10 min).
!python -m ml.foodphoto.clip_embed --model clip_b16 --workers 2

In [ ]:
# 8. ViT-L/14: the largest (~25 min).
!python -m ml.foodphoto.clip_embed --model clip_l14 --workers 2 --batch-size 128

In [ ]:
# 9. What is on Drive (each model: text.npz, train.npz, val.npz, test.npz).
import glob
for path in sorted(glob.glob(f'{DRIVE}/runs/clip_*/*.npz')):
    print(f'{os.path.getsize(path) / 1e6:7.1f} MB  {path}')